In [13]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import defaultdict
import time, json, os, re
import scipy.sparse as sp
import gurobipy as gp
from gurobipy import GRB
import json

In [14]:
from funciones import (obtener_comunas, dist, obtener_region, resultados_sampleo, ensure_dir,
                       safe_attr, parse_x_name, extraer_y_guardar_modelo, matriz_X_desde_modelo,
                       promedio_X, comparar_con_baseline, build_matrices_from_gurobi, delta_b_from_eps,
                       extraer_prob_centros)

In [15]:
from modelos import (modelo_con_limite, modelo_centros_fijos_con_limite,
                     modelo_sin_limite, modelo_con_limite_con_obj)

In [16]:
comunas = pd.read_excel('DataEEUU/data_eeuu_procesada/comunas_ia.xlsx')
distancias = pd.read_excel('DataEEUU/data_eeuu_procesada/distancias_ia.xlsx')

In [17]:
with open('DataEEUU/data_eeuu_procesada/s_nuevo_ia.txt', 'r') as dict_file:
    dict_text = dict_file.read()
    dict_s = eval(dict_text)

In [18]:
R = obtener_comunas(comunas, "iowa")
#R = obtener_comunas(comunas, "wisconsin")
#R = obtener_comunas(comunas, "pennsylvania")


In [18]:
epsilons = np.round(np.arange(0.001, 0.81, 0.01), 2)

modelo_sin_limite_ia = None
epsilon_factible = None

for eps in epsilons:
    print(f"\nProbando epsilon = {eps:.2f}")

    modelo = modelo_sin_limite(eps, R, 17, dict_s, comunas)

    if modelo is not False and modelo is not None:
        modelo_sin_limite_ia = modelo
        epsilon_factible = eps
        print(f"\nFACTIBLE con epsilon = {eps:.2f}")
        break

if modelo_sin_limite_ia is None:
    print("\nNo se encontró epsilon factible en el rango probado.")
else:
    print(f"\nMenor epsilon factible encontrado: {epsilon_factible:.2f}")


Probando epsilon = 0.00
La cantidad de centros es 17
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 21006 rows, 4556 columns and 59566 nonzeros
Model fingerprint: 0x8d82936f
Coefficient statistics:
  Matrix range     [1e+00, 2e+06]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 2e+01]
Presolve removed 1424 rows and 268 columns
Presolve time: 0.06s
Presolved: 4489 rows, 20738 columns, 50186 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.19s
Ordering time: 0.21s

Barrier statistics:
 Dense cols : 67
 Free vars  : 134
 AA' NZ     : 1.526e+05
 Factor NZ  : 4.557e+05 (roughly 14 MB of memory)
 Factor Ops : 4.757e+07 (less than 1 second per iteration)
 T

In [19]:
from gurobipy import *
from gurobipy import Model
from funciones import calcular_poblacion_total, obtener_region, pob, codigo_com_cut, codigo_cut_com

import time
from datetime import datetime
import geopandas as gpd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
from openpyxl import load_workbook
import scipy as sp
from collections import defaultdict


In [20]:

modelo_sin_limite_ia = modelo_sin_limite(0.000000001, R, 4, dict_s, comunas)

La cantidad de centros es 4
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 55464 rows, 9900 columns and 149538 nonzeros
Model fingerprint: 0xbac14250
Coefficient statistics:
  Matrix range     [1e+00, 8e+05]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 4e+00]
Presolve removed 296 rows and 99 columns
Presolve time: 0.15s
Presolved: 9801 rows, 55168 columns, 139242 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.47s
Ordering time: 0.56s

Barrier statistics:
 Dense cols : 100
 Free vars  : 100
 AA' NZ     : 4.901e+05
 Factor NZ  : 1.457e+06 (roughly 40 MB of memory)
 Factor Ops : 2.233e+08 (less than 1 second per iteration)
 Threads    : 8

          

In [21]:
modelo_sin_limite_ia.write("datos_modelo/modelo_ia.lp")

valores_ia = {v.VarName: v.X for v in modelo_sin_limite_ia.getVars()}

with open("datos_modelo/valores_ia.json", "w") as f:
    json.dump(valores_ia, f)

In [22]:
modelo_read = gp.read("datos_modelo/modelo_ia.lp")

with open("datos_modelo/valores_ia.json", "r") as f:
    valores_raw = json.load(f)
    
valores = {}
for k, v in valores_raw.items():
    valores[k.replace(" ", "_")] = v

Read LP format model from file datos_modelo/modelo_ia.lp
Reading time = 0.11 seconds
: 55464 rows, 9900 columns, 149538 nonzeros


In [23]:
def extraer_prob_centros(modelo, K, valores):
    centros_frac = []
    centros = []
    count_centros_fijados = 0
    
    for v in modelo.getVars():
        if v.VarName.startswith("centros_j") and valores[v.VarName] == 1.0:
            texto_1 = v.VarName
            comuna = texto_1[texto_1.find('[')+1 : texto_1.find(']')]
            centros.append(comuna)
            count_centros_fijados += 1

        elif v.VarName.startswith("centros_j") and valores[v.VarName] > 0.0 and valores[v.VarName] < 1.0:
            texto_1 = v.VarName
            comuna = texto_1[texto_1.find('[')+1 : texto_1.find(']')]
            centros_frac.append((comuna, valores[v.VarName]))

    top_centros_frac = sorted(centros_frac, key=lambda x: x[1], reverse=True)[0:(K-count_centros_fijados)]
    
    print(f"Hay {count_centros_fijados} centros fijados por el Modelo\n")
    print(f"Hay {len(centros_frac)} comunas con peso positivo\n")

    return count_centros_fijados, centros_frac, top_centros_frac

In [24]:
count_centros_fijados, centros_frac, top_centros_frac = extraer_prob_centros(modelo_sin_limite_ia, 4, valores)

Hay 0 centros fijados por el Modelo

Hay 88 comunas con peso positivo

